# Prepare AromatasePred Dashboard Data

This notebook rebuilds the data behind the AromatasePred Streamlit dashboard, starting from raw bioactivity records in ChEMBL:

1. Fetch raw IC50 records for human aromatase (CYP19A1, `CHEMBL1978`) from the ChEMBL REST API into `CHEMBL1978_RAW_ACTIVITIES`
2. Clean and deduplicate them into `CHEMBL1978_COMPOUNDS`, labelling compounds Active at pIC50 >= 6.5
3. Compute Lipinski Rule of Five properties with RDKit
4. Score every compound with the R model served by `QSAR_PLUMBER_SVC`
5. Write `dashboard_eda.csv`, `dashboard_ro5.csv`, `dashboard_predictions.csv`, and `chembl1978_cleaned.csv` to the app's stage

**Requirements:** run on a Container Runtime with an external access integration that allows `www.ebi.ac.uk`, `pypi.org`, and `files.pythonhosted.org` (`QSAR_NB_EAI` in `sql/setup.sql`), and with `QSAR_PLUMBER_SVC` running.

## Configuration

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DATABASE = "QSAR_DB"
SCHEMA = "PUBLIC"
SERVICE = "QSAR_PLUMBER_SVC"
STAGE = f"@{DATABASE}.{SCHEMA}.QSAR_STAGE"

TARGET_CHEMBL_ID = "CHEMBL1978"   # human aromatase (CYP19A1)
ACTIVE_THRESHOLD = 6.5            # pIC50 >= 6.5 (IC50 <= ~316 nM) is Active

session.use_database(DATABASE)
session.use_schema(SCHEMA)
print(session.get_current_database(), session.get_current_schema())

## Install RDKit

RDKit computes the Rule of Five descriptors. It installs from PyPI through the notebook's external access integration. The version is pinned because hydrogen-bond acceptor counts changed in later RDKit releases.

In [ ]:
%pip install -q rdkit==2025.09.3

## Fetch Raw Bioactivity Data

Query the ChEMBL REST API for binding-assay IC50 values reported with an exact (`=`) relation. Records are stored as-is so the cleaning step below is fully reproducible in SQL.

In [ ]:
import requests
import pandas as pd

CHEMBL_API = "https://www.ebi.ac.uk/chembl/api/data"
FIELDS = ["activity_id", "molecule_chembl_id", "canonical_smiles",
          "standard_value", "standard_units", "standard_relation", "pchembl_value"]

release = requests.get(f"{CHEMBL_API}/status.json", timeout=60).json()
print("ChEMBL release:", release["chembl_db_version"], release["chembl_release_date"])

params = {
    "target_chembl_id": TARGET_CHEMBL_ID,
    "standard_type": "IC50",
    "standard_relation": "=",
    "assay_type": "B",
    "only": ",".join(FIELDS),
    "limit": 1000,
}
records, url = [], f"{CHEMBL_API}/activity.json"
while url:
    page = requests.get(url, params=params, timeout=120).json()
    records.extend(page["activities"])
    nxt = page["page_meta"]["next"]
    url, params = (f"https://www.ebi.ac.uk{nxt}" if nxt else None), None

raw_df = pd.DataFrame(records)[FIELDS].astype("string")
raw_df.columns = [c.upper() for c in raw_df.columns]
print(f"Raw records: {len(raw_df)}")

session.write_pandas(raw_df, "CHEMBL1978_RAW_ACTIVITIES",
                     auto_create_table=True, overwrite=True, table_type="")

## Clean and Deduplicate

Keep nM records with a SMILES string and a pChEMBL value, then keep the highest pIC50 per molecule. This is the curated, non-redundant set the R model was trained on. The result is saved as the `CHEMBL1978_COMPOUNDS` table.

In [ ]:
df = session.table("CHEMBL1978_RAW_ACTIVITIES").to_pandas()   # bounded: a few thousand raw records
df.columns = [c.lower() for c in df.columns]
df = df.sort_values("activity_id", key=lambda s: pd.to_numeric(s))   # ChEMBL API order

df = df[df["standard_units"] == "nM"]
df = df.dropna(subset=["canonical_smiles", "pchembl_value"])
df = df[(df["canonical_smiles"].str.len() > 0) & (df["canonical_smiles"] != "None")].copy()
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df["pchembl_value"] = pd.to_numeric(df["pchembl_value"], errors="coerce")
df = df.dropna(subset=["pchembl_value"])

# Keep the highest pIC50 measurement for each molecule; a stable sort keeps the
# earliest ChEMBL record when measurements tie, so reruns give the same result
compounds = (
    df.sort_values("pchembl_value", ascending=False, kind="stable")
      .drop_duplicates(subset="molecule_chembl_id", keep="first")
      .reset_index(drop=True)
)
compounds["activity_class"] = (compounds["pchembl_value"] >= ACTIVE_THRESHOLD).astype(int)
compounds["activity_label"] = compounds["activity_class"].map({1: "Active", 0: "Inactive"})

table_df = compounds[["molecule_chembl_id", "canonical_smiles", "standard_value",
                      "pchembl_value", "activity_class"]].copy()
table_df.columns = [c.upper() for c in table_df.columns]
session.write_pandas(table_df, "CHEMBL1978_COMPOUNDS",
                     auto_create_table=True, overwrite=True, table_type="")
print(f"Compounds: {len(compounds)}")

In [ ]:
%%sql -r compound_summary
SELECT
    COUNT(*)                           AS N_COMPOUNDS,
    SUM(ACTIVITY_CLASS)                AS N_ACTIVE,
    COUNT(*) - SUM(ACTIVITY_CLASS)     AS N_INACTIVE,
    ROUND(MIN(PCHEMBL_VALUE), 2)       AS MIN_PIC50,
    ROUND(MAX(PCHEMBL_VALUE), 2)       AS MAX_PIC50
FROM CHEMBL1978_COMPOUNDS

The 2,751-compound training set has 1,061 actives. A newer ChEMBL release can change these counts.

About 130 molecules have more than one measurement at the same top pIC50. This notebook keeps the earliest record, so `standard_value` can differ slightly from the repository's `data/chembl1978_cleaned.csv` for those molecules. SMILES, pIC50, and activity class are identical, and the model and dashboard use only those columns.

## Rule of Five Properties

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import Crippen, Descriptors, rdMolDescriptors

RDLogger.DisableLog("rdApp.*")

ro5_rows = []
for row in compounds.itertuples():
    mol = Chem.MolFromSmiles(row.canonical_smiles)
    if mol is None:
        continue
    ro5_rows.append({
        "pchembl_value":  row.pchembl_value,
        "mw":             round(Descriptors.MolWt(mol), 2),
        "logp":           round(Crippen.MolLogP(mol), 3),
        "hbd":            rdMolDescriptors.CalcNumHBD(mol),
        "hba":            rdMolDescriptors.CalcNumHBA(mol),
        "tpsa":           round(Descriptors.TPSA(mol), 2),
        "activity_label": row.activity_label,
    })
ro5_df = pd.DataFrame(ro5_rows)
print(f"Rule of Five rows: {len(ro5_df)} ({len(compounds) - len(ro5_df)} unparsable)")

## Score Compounds With the R Model

The notebook calls the R Plumber service over its internal Snowpark Container Services DNS name, so no token or public endpoint is needed. Requests use the service-function row format, `[row_index, smiles]`.

In [ ]:
svc = session.sql(f"SHOW SERVICES LIKE '{SERVICE}' IN SCHEMA {DATABASE}.{SCHEMA}").collect()
predict_url = f"http://{svc[0]['dns_name']}:5000/predict"
print("Service:", predict_url)

BATCH_SIZE = 50
smiles = compounds["canonical_smiles"].tolist()
predictions = [None] * len(smiles)
for start in range(0, len(smiles), BATCH_SIZE):
    batch = smiles[start:start + BATCH_SIZE]
    resp = requests.post(predict_url, json={"data": [[i, s] for i, s in enumerate(batch)]}, timeout=120)
    resp.raise_for_status()
    for idx, pic50, *_ in resp.json()["data"]:
        predictions[start + idx] = pic50

pred_df = pd.DataFrame({
    "actual_pic50":   compounds["pchembl_value"],
    "pred_pic50":     pd.to_numeric(pd.Series(predictions), errors="coerce").round(3),
    "activity_label": compounds["activity_label"],
}).dropna(subset=["pred_pic50"]).reset_index(drop=True)
print(f"Predictions: {len(pred_df)} of {len(compounds)}")

## Write Dashboard Files to the Stage

In [ ]:
import os
import tempfile

outputs = {
    "dashboard_eda.csv":         compounds[["pchembl_value", "activity_label"]],
    "dashboard_ro5.csv":         ro5_df,
    "dashboard_predictions.csv": pred_df,
    "chembl1978_cleaned.csv":    compounds[["molecule_chembl_id", "canonical_smiles", "standard_value",
                                            "pchembl_value", "activity_class"]],
}

out_dir = tempfile.mkdtemp()
for name, frame in outputs.items():
    path = os.path.join(out_dir, name)
    frame.to_csv(path, index=False)
    session.file.put(path, STAGE, auto_compress=False, overwrite=True)
    print(f"{name:28s} {len(frame):5d} rows")

In [ ]:
%%sql -r stage_files
LIST @QSAR_STAGE PATTERN = '.*(dashboard_|chembl1978_cleaned).*'

## Next Steps

Restart the Streamlit app to pick up the new files; its data loaders cache results for an hour. To retrain the R model on this data, download `chembl1978_cleaned.csv` from the stage into `data/` and follow the training steps in the guide.